In [ ]:
# Imports
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import split, col
from pyspark.sql.types import IntegerType

# Create spark session
spark = SparkSession.builder.getOrCreate()

# Read in database
df = spark.read.text('trust.txt')
df1 = spark.read.text('ratings.txt')

# Name columns
df = df.withColumn('Truster', split(df['value'], ' ').getItem(0)) \
       .withColumn('Trustee', split(df['value'], ' ').getItem(1)) \
       .withColumn('Trust Value', split(df['value'], ' ').getItem(2))

df1 = df1.withColumn('Truster 1', split(df1['value'], ' ').getItem(0)) \
        .withColumn('Movie', split(df1['value'], ' ').getItem(1)) \
        .withColumn('Rating', split(df1['value'], ' ').getItem(2))

# Drop original column
df = df.drop(col('value'))
df1 = df1.drop(col('value'))

# Join dataframes
FilmTrust = df.join(df1,df['Truster'] == df1['Truster 1'])

# Drop duplicate column
FilmTrust = FilmTrust.drop(col('Truster 1'))

# Makes columns integer
FilmTrust = FilmTrust.withColumn("Truster", FilmTrust["Truster"].cast(IntegerType()))
FilmTrust = FilmTrust.withColumn("Movie", FilmTrust["Movie"].cast(IntegerType()))
FilmTrust = FilmTrust.withColumn("Rating", FilmTrust["Rating"].cast(IntegerType()))

# Drop null
FilmTrust = FilmTrust.na.drop() 

# Drop duplicates
FilmTrust = FilmTrust.dropDuplicates()

# Drop Outliers
FilmTrust = FilmTrust.filter('Truster<1509')

FilmTrust.show()

In [ ]:
from pyspark.sql.functions import asc, desc, count, when, isnan

# Summary of rating values
FilmTrust.select('Rating').summary().show()

# Top rated movies
FilmTrust.select(FilmTrust.Movie,FilmTrust.Rating).sort(FilmTrust.Rating.desc()).show()

# Lower rated movies
FilmTrust.select(FilmTrust.Movie,FilmTrust.Rating).sort(FilmTrust.Rating.asc()).show()

# Number of ratings per user
FilmTrust.groupBy('Truster').count().show()

# Number of ratings per movie
FilmTrust.groupBy('Movie').count().show()

In [ ]:
# Imports
from pyspark.ml.evaluation import RegressionEvaluator 
from pyspark.ml.recommendation import ALS

# Split data
train_data, test_data = FilmTrust.randomSplit([0.8, 0.2])

# Build recommender model
als = ALS(maxIter=5, regParam=0.01, userCol='Truster', itemCol='Movie', ratingCol='Rating') 

# Fit
model = als.fit(train_data)

# Evaluate
predictions = model.transform(test_data)
predictions.show()

# Drop null
predictions = predictions.na.drop() 

# RMSE
evaluator = RegressionEvaluator(metricName="rmse", labelCol="Rating",predictionCol="prediction") 
rmse = evaluator.evaluate(predictions) 
print("Root-mean-square error = " + str(rmse))